In [0]:
# =============================================================================
# DQ Validation - ALL states
#
# The single Active DQ notebook. Merges the two previous notebooks:
#   * report_dq_state_of_play   -> per-state STATE OF PLAY (segmented / valid / quarantined)
#   * failed_cases_by_rule      -> per-case FAILURE ATTRIBUTION (which rules failed, FALSE vs NULL)
# and adds a run to the central Spark tables so DQ is tracked over time like the other suites.
#
# Reads the per-state stg_main_<state>_validation tables ONCE each (both analyses come from the
# same read) and evaluates the LIVE dev DQ rules against them.
#
# Outputs (one timestamped folder):
#   StateOfPlay    - per state: segmented, valid, quarantined, quarantine %, rule severity counts
#   FailedCases    - one row per quarantined case + which rules were FALSE / NULL
#   RuleCounts     - each FALSE rule + cases/states it hit          <- the headline attribution
#   NullRuleCounts - rules that ABSTAINED (NULL) - not failures
#   RuleMatrix     - rule x state failure grid
#   Detail         - per state+rule passed/total/failed/severity
#   Summary        - funnel totals + attribution integrity (genuine / NULL-only / unattributed)
#   MissingTables / SkippedRules - what could not be read / evaluated
#   + one run in test_automation_runs2 and per-state/per-rule rows in test_automation_results2
#
# NOTE: reports on the PIPELINE's DQ outcome (reads is_valid + evaluates the rules); it does not
# run the DLT job itself. Archive DQ is deliberately OUT of scope (archive has no quarantine table).
#
# A DQ rule is a SQL expression, so it has THREE outcomes - not two:
#   TRUE  = rule satisfied.
#   FALSE = genuine violation. This is what DLT counts and what quarantined the case.
#   NULL  = the rule gave NO verdict. A NULL operand made the SQL unknown - e.g.
#           array_contains(valid_categoryIdList, 38) is NULL when the list is NULL, and
#           NULL AND TRUE = NULL - or a CASE matched no WHEN. DLT treats NULL as a PASS,
#           so a NULL rule did NOT cause the quarantine.
# Counting NULL as a failure would over-attribute rules that simply do not apply to the case,
# which is why FALSE and NULL are tallied separately (RuleCounts vs NullRuleCounts).
# NULL-only cases = quarantined by the pipeline, but no rule is FALSE when re-evaluated now
# => the data or the rules have drifted since is_valid was written (a staleness signal).
# =============================================================================

In [0]:
import os, datetime, base64, json, uuid
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

RESULTS_DIR = "/Workspace/Users/" + spark.sql("select current_user()").first()[0] + "/Results/DQ_Validation_All_States"
run_by_automation_name = "DQ_Validation_Automation"
test_runs_table = "test_automation_runs2"
test_results_table = "test_automation_results2"

STATE_MAIN = {
    "appealSubmitted":               "hive_metastore.appealsubmitted_gold.stg_main_appeal_submitted_validation",
    "paymentPending":                "hive_metastore.paymentPending_gold.stg_main_paymentPending_validation",
    "awaitingRespondentEvidence(a)": "hive_metastore.awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation",
    "awaitingRespondentEvidence(b)": "hive_metastore.awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation",
    "caseUnderReview":               "hive_metastore.caseunderreview_gold.stg_main_case_under_review_validation",
    "decided(a)":                    "hive_metastore.decideda_gold.stg_main_decided_a_validation",
    "decided(b)":                    "hive_metastore.decidedb_gold.stg_main_decided_b_validation",
    "decision":                      "hive_metastore.decision_gold.stg_main_decision_validation",
    "ended":                         "hive_metastore.ended_gold.stg_main_ended_validation",
    "ftpaDecided":                   "hive_metastore.ftpadecided_gold.stg_main_ftpadecided_validation",
    "ftpaSubmitted(a)":              "hive_metastore.ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation",
    "ftpaSubmitted(b)":              "hive_metastore.ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation",
    "listing":                       "hive_metastore.listing_gold.stg_main_listing_validation",
    "prepareForHearing":             "hive_metastore.prepareforhearing_gold.stg_main_prepare_for_hearing_validation",
    "reasonsForAppealSubmitted":     "hive_metastore.reasonsforappealsubmitted_gold.stg_main_reasons_for_appeal_submitted_validation",
    "remitted":                      "hive_metastore.remitted_gold.stg_main_remitted_validation",
}

# Always read the LATEST dev DQ rules live (no frozen snapshot). Handles BOTH plain .py files AND
# Databricks notebooks (read via the Workspace export API). Maps rules by class name.
DQ_RULES_DIR = "/Workspace/live/ACTIVE/APPEALS/shared_functions/dq_rules"
_STATE_CLASS = {
    "paymentPendingDetained": "paymentPendingDetainedDQRules", "paymentPending": "paymentPendingDQRules",
    "appealSubmitted": "appealSubmittedDQRules",
    "awaitingRespondentEvidence(a)": "awaitingEvidenceRespondentADQRules",
    "awaitingRespondentEvidence(b)": "awaitingEvidenceRespondentBDQRules",
    "caseUnderReview": "caseUnderReviewDQRules", "reasonsForAppealSubmitted": "reasonsForAppealSubmittedDQRules",
    "listing": "listingDQRules", "prepareForHearing": "prepareForHearingDQRules", "decision": "decisionDQRules",
    "decided(a)": "decidedADQRules", "decided(b)": "decidedBDQRules", "ftpaSubmitted(a)": "ftpaSubmittedADQRules",
    "ftpaSubmitted(b)": "ftpaSubmittedBDQRules", "ftpaDecided": "ftpaDecidedDQRules", "ended": "endedDQRules",
    "remitted": "remittedDQRules",
}
_PREV = {
    "paymentPending": "paymentPendingDetained", "appealSubmitted": "paymentPending",
    "awaitingRespondentEvidence(a)": "appealSubmitted", "awaitingRespondentEvidence(b)": "awaitingRespondentEvidence(a)",
    "caseUnderReview": "awaitingRespondentEvidence(b)", "reasonsForAppealSubmitted": "awaitingRespondentEvidence(b)",
    "listing": "awaitingRespondentEvidence(b)", "prepareForHearing": "listing", "decision": "prepareForHearing",
    "decided(a)": "decision", "ftpaSubmitted(a)": "decided(a)", "ftpaSubmitted(b)": "ftpaSubmitted(a)",
    "ftpaDecided": "ftpaSubmitted(b)", "remitted": "ftpaDecided", "ended": "ftpaSubmitted(a)", "decided(b)": "ftpaSubmitted(a)",
}
_LANDING = [s for s in _STATE_CLASS if s != "paymentPendingDetained"]

class _StubBase:
    def get_checks(self, checks=None):
        return checks or {}

def _checks_from_source(src):
    keep = [ln for ln in src.splitlines() if not ln.strip().startswith("from .dq_rules import")]
    ns = {"DQRulesBase": _StubBase, "ABC": object, "abstractmethod": (lambda f: f)}
    exec(chr(10).join(keep), ns)
    out = {}
    for v in list(ns.values()):
        if isinstance(v, type) and issubclass(v, _StubBase) and v is not _StubBase:
            out[v.__name__] = dict(v().get_checks({}))
    return out

def _via_files(d):
    found = {}
    try:
        names = os.listdir(d)
    except Exception:
        return found
    for f in sorted(names):
        if f.endswith("_dq_rules.py") and f != "dq_rules.py":
            try:
                found.update(_checks_from_source(open(os.path.join(d, f), encoding="utf-8").read()))
            except Exception as e:
                print(f"  skip {f}: {str(e)[:60]}")
    return found

def _via_workspace(d):
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    try:
        from databricks.sdk.service.workspace import ExportFormat
        _fmt = ExportFormat.SOURCE
    except Exception:
        _fmt = "SOURCE"
    found = {}
    cands = [d[len("/Workspace"):] if d.startswith("/Workspace/") else d, d]
    for api in cands:
        try:
            objs = list(w.workspace.list(api))
        except Exception:
            continue
        for o in objs:
            name = str(o.path).rstrip("/").split("/")[-1]
            if name.endswith("_dq_rules") and name != "dq_rules":
                try:
                    r = w.workspace.export(o.path, format=_fmt)
                    found.update(_checks_from_source(base64.b64decode(r.content).decode("utf-8")))
                except Exception as e:
                    print(f"  skip {name}: {str(e)[:60]}")
        if found:
            print(f"  (read {len(found)} rule classes via Workspace API from {api})")
            return found
    return found

def _flow(state):
    chain = [state]
    while _PREV.get(chain[0]):
        chain.insert(0, _PREV[chain[0]])
    return chain

def _load_dq_rules():
    classes = _via_files(DQ_RULES_DIR)
    if len(classes) < 10:
        classes = _via_workspace(DQ_RULES_DIR) or classes
    missing = [c for c in _STATE_CLASS.values() if c not in classes]
    if missing:
        print("  WARNING missing rule classes:", missing)
    sc = {st: classes.get(cls, {}) for st, cls in _STATE_CLASS.items()}
    _all, _sr, _state_checks = {}, {}, {}
    for st in _LANDING:
        merged = {}
        for s in _flow(st):
            merged = merged | sc.get(s, {})
        _sr[st] = sorted(merged.keys())
        _state_checks[st] = merged   # per-state dict KEEPS the correct rule variant; evaluate against this,
        _all.update(merged)          # NOT ALL_CHECKS[name] (a global last-writer that collapses same-named rules)
    print(f"DQ rules loaded live: {len(_all)} checks across {len(_sr)} states")
    return {"ALL_CHECKS": _all, "STATE_RULES": _sr, "STATE_CHECKS": _state_checks}
_RULES = _load_dq_rules()
ALL_CHECKS = _RULES["ALL_CHECKS"]
STATE_RULES = _RULES["STATE_RULES"]

cfg = spark.read.option("multiline", "true").json("dbfs:/configs/config.json").first()
env = cfg["env"].strip().lower(); lz = cfg["lz_key"].strip().lower(); kv = "ingest" + lz + "-meta002-" + env
cid = dbutils.secrets.get(kv, "SERVICE-PRINCIPLE-CLIENT-ID")
sec = dbutils.secrets.get(kv, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tid = dbutils.secrets.get(kv, "SERVICE-PRINCIPLE-TENANT-ID")
for acct in ["ingest" + lz + "curated" + env, "ingest" + lz + "raw" + env]:
    spark.conf.set("fs.azure.account.auth.type." + acct + ".dfs.core.windows.net", "OAuth")
    spark.conf.set("fs.azure.account.oauth.provider.type." + acct + ".dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set("fs.azure.account.oauth2.client.id." + acct + ".dfs.core.windows.net", cid)
    spark.conf.set("fs.azure.account.oauth2.client.secret." + acct + ".dfs.core.windows.net", sec)
    spark.conf.set("fs.azure.account.oauth2.client.endpoint." + acct + ".dfs.core.windows.net", "https://login.microsoftonline.com/" + tid + "/oauth2/token")

In [0]:
# =============================================================================
# ONE pass over the per-state validation tables -> BOTH analyses
#   (a) STATE OF PLAY   : segmented / valid / quarantined + rule severity
#   (b) FAILURE ATTRIB. : per quarantined case, which rules were FALSE vs NULL
# =============================================================================
run_start_datetime = datetime.datetime.now()
STAMP = run_start_datetime.strftime("%Y%m%d_%H%M%S")
OUT = RESULTS_DIR + "/" + STAMP
os.makedirs(OUT, exist_ok=True)

MISSING_TABLES = []   # (table, reason) for anything that couldn't be read - reported at the end
SKIPPED_RULES = []    # (state, rule, reason) rules not evaluable against the table's columns
                      # -> these explain quarantined cases showing n_failed_rules = 0
state_rows = []       # (state, segmented, valid, quarantined, rules_reject_100, rules_partial)
rule_states = {}      # rule -> {state: (passed, total)}
pdf_parts = []        # per-state failed-case frames (materialised immediately)

def table_ok(t):
    """Resolvable-table check that never raises. Records + announces misses so the
    rest of the states still run when tables have been dropped (e.g. pre dry-run)."""
    try:
        if spark.catalog.tableExists(t):
            return True
        reason = "NOT FOUND"
    except Exception as e:
        reason = type(e).__name__ + ": " + str(e).splitlines()[0][:140]
    MISSING_TABLES.append((t, reason))
    print("  [skip] table unavailable (" + reason + "): " + t)
    return False

for STATE, TABLE in STATE_MAIN.items():
    if not table_ok(TABLE):
        state_rows.append((STATE, None, None, None, 0, 0))
        continue
    try:
        df = spark.table(TABLE).cache()
        total = df.count()
    except Exception as e:
        MISSING_TABLES.append((TABLE, type(e).__name__ + ": " + str(e).splitlines()[0][:140]))
        print("  [skip] read failed: " + TABLE)
        state_rows.append((STATE, None, None, None, 0, 0))
        continue
    if "CaseNo" not in df.columns or "is_valid" not in df.columns:
        MISSING_TABLES.append((TABLE, "missing CaseNo/is_valid column"))
        print("  [skip] no CaseNo/is_valid column: " + TABLE)
        state_rows.append((STATE, None, None, None, 0, 0))
        df.unpersist()
        continue

    rules = dict(_RULES["STATE_CHECKS"].get(STATE, {}))   # per-state SQL (correct variant per state)
    ok = []
    for n in rules:
        try:
            df.select(expr(rules[n])).schema
            ok.append(n)
        except Exception as e:
            SKIPPED_RULES.append((STATE, n, str(e).splitlines()[0][:180]))

    # ---------- (a) STATE OF PLAY ----------
    counts = df.agg(*[F.sum(F.when(expr(rules[n]), 1).otherwise(0)).alias("c%d" % i)
                      for i, n in enumerate(ok)]).first() if ok else None
    n100, npart = 0, 0
    for i, n in enumerate(ok):
        p = (counts["c%d" % i] if counts is not None else 0) or 0
        if p < total:
            rule_states.setdefault(n, {})[STATE] = (p, total)
            if p == 0:
                n100 += 1
            else:
                npart += 1
    valid = df.filter(col("is_valid") == True).count()
    state_rows.append((STATE, total, valid, total - valid, n100, npart))

    # ---------- (b) FAILURE ATTRIBUTION ----------
    # FALSE = a genuine violation - what the DLT job counts and what quarantined the case.
    # NULL  = the rule gave NO verdict: a NULL operand made the SQL unknown (e.g.
    #         array_contains(valid_categoryIdList, 38) with a NULL list, then NULL AND TRUE = NULL),
    #         or a CASE matched no WHEN. DLT treats NULL as a PASS, so it did NOT quarantine anything.
    # NULL must NOT be counted as a failure or it over-attributes rules that simply do not apply
    # to the case (e.g. valid_ftpaApplicationDeadline on a case with no FTPA application).
    false_cols = [F.when(expr(rules[n]).cast("boolean") == F.lit(False), F.lit(n)) for n in ok]
    null_cols = [F.when(expr(rules[n]).cast("boolean").isNull(), F.lit(n)) for n in ok]
    failed = df.where(~F.coalesce(col("is_valid").cast("boolean"), F.lit(False)))
    farr = F.array(*false_cols) if false_cols else F.array().cast("array<string>")
    narr = F.array(*null_cols) if null_cols else F.array().cast("array<string>")
    failed = (failed
              .withColumn("_ff", farr).withColumn("_ff", F.expr("filter(_ff, x -> x is not null)"))
              .withColumn("_nn", narr).withColumn("_nn", F.expr("filter(_nn, x -> x is not null)")))
    d = failed.select(
        F.lit(STATE).alias("state"),
        F.trim(col("CaseNo")).alias("CaseNo"),
        col("is_valid"),
        F.size("_ff").alias("n_failed_rules"),
        F.concat_ws(", ", col("_ff")).alias("failed_rules"),
        F.size("_nn").alias("n_null_rules"),
        F.concat_ws(", ", col("_nn")).alias("null_rules"))
    try:
        _part = d.toPandas()   # action happens here, right after reading THIS table (tiny race window)
        if len(_part):
            pdf_parts.append(_part)
    except Exception as e:
        MISSING_TABLES.append((TABLE, "action failed (schema changed mid-run?): " + str(e).splitlines()[0][:110]))
        print("  [skip] action failed for " + TABLE + " - re-run after the pipeline finishes writing it")
    df.unpersist()

In [0]:
# =============================================================================
# Build the sheets
# =============================================================================
STATES = [r[0] for r in state_rows if r[1] is not None]

# --- StateOfPlay: per state segmented / valid / quarantined (the funnel the reports quote) ---
sop = []
for STATE, total, valid, invalid, n100, npart in state_rows:
    if total is None:
        sop.append([STATE, None, None, None, None, 0, 0])
        continue
    sop.append([STATE, total, valid, invalid, round(100.0 * invalid / max(total, 1), 1), n100, npart])
state_of_play_df = pd.DataFrame(sop, columns=["state", "segmented", "valid", "quarantined",
                                              "quarantined_pct", "rules_reject_100", "rules_partial"])
TOT_SEG = int(state_of_play_df["segmented"].fillna(0).sum())
TOT_VALID = int(state_of_play_df["valid"].fillna(0).sum())
TOT_QUAR = int(state_of_play_df["quarantined"].fillna(0).sum())

# --- FailedCases (one row per quarantined case = is_valid not True) ---
pdf = (pd.concat(pdf_parts, ignore_index=True) if pdf_parts
       else pd.DataFrame(columns=["state", "CaseNo", "is_valid", "n_failed_rules", "failed_rules", "n_null_rules", "null_rules"]))
pdf = pdf.sort_values(["state", "CaseNo"]) if len(pdf) else pdf

# --- RuleCounts (each FALSE rule + how many cases / states it hit) ---
if len(pdf):
    exploded = pdf.assign(rule=pdf["failed_rules"].str.split(", ")).explode("rule")
    exploded = exploded[exploded["rule"].notna() & (exploded["rule"].astype(str).str.len() > 0)]
    rule_counts = (exploded.groupby("rule")
                   .agg(cases_failed=("CaseNo", "nunique"), states_affected=("state", "nunique"))
                   .reset_index()
                   .sort_values(["cases_failed", "rule"], ascending=[False, True]))
else:
    rule_counts = pd.DataFrame(columns=["rule", "cases_failed", "states_affected"])

# --- NullRuleCounts (rules that returned NULL = no verdict; DLT passes these, so NOT failures) ---
if len(pdf):
    nexp = pdf.assign(rule=pdf["null_rules"].str.split(", ")).explode("rule")
    nexp = nexp[nexp["rule"].notna() & (nexp["rule"].astype(str).str.len() > 0)]
    null_counts = (nexp.groupby("rule")
                   .agg(cases_null=("CaseNo", "nunique"), states_affected=("state", "nunique"))
                   .reset_index()
                   .sort_values(["cases_null", "rule"], ascending=[False, True]))
else:
    null_counts = pd.DataFrame(columns=["rule", "cases_null", "states_affected"])

# label cases with no FALSE rule. Done AFTER the tallies so the note doesn't pollute them.
if len(pdf):
    m0 = pdf["n_failed_rules"] == 0
    pdf.loc[m0 & (pdf["n_null_rules"] > 0), "failed_rules"] = "(no FALSE rule - NULL/abstained only; see null_rules)"
    pdf.loc[m0 & (pdf["n_null_rules"] == 0), "failed_rules"] = "(no evaluable failing rule - see SkippedRules)"

# --- RuleMatrix / Detail (rule x state) ---
matrix = []
for rule in sorted(rule_states):
    hits = rule_states[rule]
    row = {"rule": rule}
    n100 = 0
    for st in STATES:
        if st in hits:
            p, t = hits[st]
            row[st] = t - p
            if p == 0:
                n100 += 1
        else:
            row[st] = 0
    row["states_failing"] = len(hits)
    row["states_reject_100"] = n100
    matrix.append(row)
matrix_df = pd.DataFrame(matrix, columns=["rule"] + STATES + ["states_failing", "states_reject_100"])
if len(matrix_df):
    matrix_df = matrix_df.sort_values(["states_reject_100", "states_failing"], ascending=False)

detail = []
for rule in sorted(rule_states):
    for st in rule_states[rule]:
        p, t = rule_states[rule][st]
        detail.append([st, rule, p, t, t - p, round(100.0 * (t - p) / max(t, 1), 1), "REJECTS_100" if p == 0 else "partial"])
detail_df = pd.DataFrame(detail, columns=["state", "rule", "passed", "total", "failed", "fail_pct", "severity"])
if len(detail_df):
    detail_df = detail_df.sort_values(["state", "severity", "failed"], ascending=[True, True, False])

# --- Summary: funnel + attribution integrity ---
n_genuine = int((pdf["n_failed_rules"] > 0).sum()) if len(pdf) else 0
n_nullonly = int(((pdf["n_failed_rules"] == 0) & (pdf["n_null_rules"] > 0)).sum()) if len(pdf) else 0
n_unattr = int(((pdf["n_failed_rules"] == 0) & (pdf["n_null_rules"] == 0)).sum()) if len(pdf) else 0
summary = pd.DataFrame([
    ("states_reported",                          len(STATES)),
    ("total_segmented",                          TOT_SEG),
    ("total_valid_gold",                         TOT_VALID),
    ("total_quarantined_cases",                  TOT_QUAR),
    ("quarantined_pct",                          round(100.0 * TOT_QUAR / max(TOT_SEG, 1), 1)),
    ("cases_with_a_FALSE_rule (genuine)",        n_genuine),
    ("cases_NULL_only (no FALSE rule)",          n_nullonly),
    ("cases_unattributed (no FALSE/NULL rule)",  n_unattr),
    ("unique_FALSE_rules",                       int(rule_counts.shape[0])),
    ("unique_NULL_rules",                        int(null_counts.shape[0])),
    ("most_triggered_FALSE_rule",                rule_counts.iloc[0]["rule"] if len(rule_counts) else None),
    ("most_triggered_FALSE_rule_cases",          int(rule_counts.iloc[0]["cases_failed"]) if len(rule_counts) else 0),
    ("states_with_failures",                     int(pdf["state"].nunique()) if len(pdf) else 0),
    ("tables_missing_or_skipped",                len(MISSING_TABLES)),
    ("rules_skipped_not_evaluable",              len(SKIPPED_RULES)),
], columns=["metric", "value"])

missing_df = pd.DataFrame(MISSING_TABLES, columns=["table", "reason"])
skipped_df = pd.DataFrame(SKIPPED_RULES, columns=["state", "rule", "reason"]).drop_duplicates()

# CSVs (always) + one multi-sheet Excel
state_of_play_df.to_csv(OUT + "/state_of_play.csv", index=False)
pdf.to_csv(OUT + "/failed_cases_by_rule.csv", index=False)
rule_counts.to_csv(OUT + "/rule_counts.csv", index=False)
null_counts.to_csv(OUT + "/null_rule_counts.csv", index=False)
summary.to_csv(OUT + "/summary.csv", index=False)
missing_df.to_csv(OUT + "/missing_tables.csv", index=False)
skipped_df.to_csv(OUT + "/skipped_rules.csv", index=False)

XLSX = OUT + "/dq_validation_all_states.xlsx"
try:
    import openpyxl  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
try:
    with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
        summary.to_excel(w, sheet_name="Summary", index=False)
        state_of_play_df.to_excel(w, sheet_name="StateOfPlay", index=False)
        pdf.to_excel(w, sheet_name="FailedCases", index=False)
        rule_counts.to_excel(w, sheet_name="RuleCounts", index=False)
        null_counts.to_excel(w, sheet_name="NullRuleCounts", index=False)
        matrix_df.to_excel(w, sheet_name="RuleMatrix", index=False)
        detail_df.to_excel(w, sheet_name="Detail", index=False)
        missing_df.to_excel(w, sheet_name="MissingTables", index=False)
        skipped_df.to_excel(w, sheet_name="SkippedRules", index=False)
    print("xlsx:", XLSX)
except Exception as e:
    print("xlsx engine failed, CSVs only:", e)

In [0]:
# =============================================================================
# Central results tables - one run_id per execution so DQ is tracked over time
# (same pattern as data_type_validation_all_states / ccd_validation_test)
# =============================================================================
from dataclasses import dataclass

@dataclass
class TestResult:
    test_field: str = ""
    status: str = ""
    message: str = ""
    test_from_state: str = ""
    test_name: str = ""

all_test_results = []

# 1) headline
all_test_results.append(TestResult(
    ">>> DQ SUMMARY <<<", "PASS" if TOT_QUAR == 0 else "FAIL",
    f"{TOT_QUAR} of {TOT_SEG} segmented cases quarantined ({round(100.0*TOT_QUAR/max(TOT_SEG,1),1)}%); "
    f"{TOT_VALID} valid gold. Attribution: {n_genuine} genuine (a FALSE rule), {n_nullonly} NULL-only, "
    f"{n_unattr} unattributed.", "all_states", "DQ summary (headline)"))
# 2) per state
for STATE, total, valid, invalid, n100, npart in state_rows:
    if total is None:
        all_test_results.append(TestResult(STATE, "NO_DATA", "validation table unavailable - see MissingTables",
                                           STATE, "DQ state of play"))
        continue
    all_test_results.append(TestResult(
        STATE, "PASS" if invalid == 0 else "FAIL",
        f"segmented {total} | valid {valid} | quarantined {invalid} ({round(100.0*invalid/max(total,1),1)}%)",
        STATE, "DQ state of play"))
# 3) per FALSE rule (the attribution signal we trend over time)
for _, r in rule_counts.iterrows():
    all_test_results.append(TestResult(
        str(r["rule"]), "FAIL",
        f"{int(r['cases_failed'])} case(s) failed this rule across {int(r['states_affected'])} state(s).",
        "all_states", "DQ rule failure (FALSE)"))
# 4) per NULL rule (abstained - informational)
for _, r in null_counts.iterrows():
    all_test_results.append(TestResult(
        str(r["rule"]), "NO_DATA",
        f"{int(r['cases_null'])} case(s) where this rule abstained (NULL) across {int(r['states_affected'])} state(s).",
        "all_states", "DQ rule abstained (NULL)"))

def run_and_result_table_schemas():
    runs_schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("run_user", StringType(), True),
        StructField("run_start_datetime", TimestampType(), True),
        StructField("run_end_datetime", TimestampType(), True),
        StructField("run_by_automation_name", StringType(), True),
        StructField("run_tag", StringType(), True),
        StructField("run_status", StringType(), True),
        StructField("state_under_test", StringType(), True),
    ])
    results_schema = StructType([
        StructField("result_id", StringType(), True),
        StructField("run_id", StringType(), True),   # FK to test_runs
        StructField("test_name", StringType(), True),
        StructField("test_field", StringType(), True),
        StructField("test_from_state", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
    ])
    return runs_schema, results_schema

def create_run(run_user, run_start_datetime, run_end_datetime, run_by_automation_name, run_tag, run_status, state_under_test):
    try:
        run_id = str(uuid.uuid4())
        runs_schema, _ = run_and_result_table_schemas()
        df = spark.createDataFrame(
            [(run_id, run_user, run_start_datetime, run_end_datetime, run_by_automation_name, run_tag, run_status, state_under_test)],
            runs_schema)
        df.write.option("mergeSchema", "true").mode("append").saveAsTable(test_runs_table)
        print(f"Created New Run Record. id : {run_id}")
        return run_id
    except Exception as e:
        print(f"Failed to Create New Run Record. error : {str(e)}")
        return None

def create_results(run_id):
    try:
        _, results_schema = run_and_result_table_schemas()
        rows = []
        for result in all_test_results:
            rows.append((
                str(uuid.uuid4()), run_id,
                str(getattr(result, "test_name", "") or ""),
                str(getattr(result, "test_field", "") or ""),
                str(getattr(result, "test_from_state", "") or ""),
                str(getattr(result, "status", "") or ""),
                str(getattr(result, "message", "") or ""),
            ))
        print(f"Built {len(rows)} rows, writing to table...")
        spark.createDataFrame(rows, results_schema).write.option("mergeSchema", "true").mode("append").saveAsTable(test_results_table)
        print(f"Created {len(rows)} result records")
    except Exception as e:
        print(f"create_results FAILED: {e}")
        import traceback; traceback.print_exc()

run_user = spark.sql("select current_user()").first()[0]
run_end_datetime = datetime.datetime.now()
total_count = len(all_test_results)
pass_count = sum(1 for r in all_test_results if r.status == "PASS")
fail_count = sum(1 for r in all_test_results if r.status == "FAIL")
run_status = "PASS" if TOT_QUAR == 0 else "FAIL"
run_tag = "DQ validation - all states"

run_id = create_run(run_user, run_start_datetime, run_end_datetime, run_by_automation_name,
                    run_tag, run_status, ", ".join(STATES))
if run_id is not None:
    create_results(run_id)
else:
    print("Failed to Create a Run, No results have been submitted to spark tables")

In [0]:
# =============================================================================
# Where the results have been put
# =============================================================================
print("=" * 70)
print("DQ VALIDATION - SUMMARY (all states)")
print("=" * 70)
print(f"Segmented: {TOT_SEG} | Valid gold: {TOT_VALID} | Quarantined: {TOT_QUAR} "
      f"({round(100.0*TOT_QUAR/max(TOT_SEG,1),1)}%)")
print(f"Attribution: {n_genuine} genuine (FALSE rule) + {n_nullonly} NULL-only + {n_unattr} unattributed")
print(f"Unique FALSE rules: {int(rule_counts.shape[0])} | Unique NULL rules: {int(null_counts.shape[0])}")
print("-" * 70)
print(f"{'State':34}{'Segmented':>10}{'Valid':>8}{'Quar':>7}{'Quar%':>8}")
for _, r in state_of_play_df.iterrows():
    if pd.isna(r["segmented"]):
        print(f"{r['state']:34}{'(no table)':>33}")
        continue
    print(f"{r['state']:34}{int(r['segmented']):>10}{int(r['valid']):>8}{int(r['quarantined']):>7}{r['quarantined_pct']:>8}")
print("-" * 70)
print("Top FALSE rules:")
for _, r in rule_counts.head(10).iterrows():
    print(f"   {r['rule']} ({int(r['cases_failed'])} cases / {int(r['states_affected'])} states)")
print("=" * 70)
print("================ RESULTS LOCATIONS ================")
print(f"Overall: {run_status}  |  Results rows: {total_count}  Pass: {pass_count}  Fail: {fail_count}")
print(f"Folder        : {OUT}")
print(f"XLSX          : {XLSX}")
print(f"Spark runs    : {test_runs_table}  (run_id = {run_id})")
print(f"Spark results : {test_results_table}  (filter on run_id = {run_id})")
if MISSING_TABLES:
    print("\n" + "!" * 60)
    print("MISSING / UNREADABLE TABLES (%d) - skipped, rest still ran:" % len(MISSING_TABLES))
    for t, reason in MISSING_TABLES:
        print("  - %-70s %s" % (t, reason))
    print("!" * 60)
else:
    print("All tables resolved - no missing tables.")